<a id="inicio"></a>

# Sentiment Analysis at Scale — Part 2: Inference

## Objective

Load the serialized sentiment-analysis pipeline produced in Part 1 and run batch scoring against the held-out test set stored in S3.

## Methodology

- **Model loading**: Restore the Spark ML pipeline saved to S3
- **Batch scoring**: Apply the pipeline to the test dataset
- **Evaluation**: Compute AUC to confirm the model generalizes to unseen reviews

<div class="alert alert-block alert-info">

### ℹ️ Note on cell outputs

The cell outputs visible in this notebook were captured during the **original run** and are shown here as a visual reference for portfolio review. Some artifacts may still appear in Spanish because the outputs predate the English code translation.

**To re-run locally:**

1. This notebook runs on AWS Glue (Spark) — a local re-run requires a Spark environment (PySpark 3.x) and AWS credentials with S3 access.
2. Set your AWS credentials in a `.env` file before launching Jupyter Lab (the Dockerfile + docker-compose.yml expect it).
3. Download the Amazon Reviews Parquet dataset (`amazon-reviews-pds-parquet`) to your working environment.
4. The training notebook writes a trained model to S3; the test notebook loads it back.

</div>

---

<a id="indice"></a>
## Contents

* [1. Batch Inference with the Serialized Model](#section1)

---

Environment setup — enter your AWS Academy credentials in the `.env` file before launching Jupyter Lab.

In [5]:
%iam_role arn:aws:iam::270830193283:role/LabRole
%region us-east-1
%number_of_workers 2
%idle_timeout 30

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
It looks like there is a newer version of the kernel available. The latest version is 1.0.9 and you have 1.0.4 installed.
Please run `pip install --upgrade aws-glue-sessions` to upgrade your kernel
Current iam_role is None
iam_role has been set to arn:aws:iam::270830193283:role/LabRole.
Previous region: None
Setting new region to: us-east-1
Region is set to: us-east-1
Previous number of workers: None
Setting new number of workers to: 2
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.


In [1]:
import os
s3_bucket = f"s3://{os.environ['WORK_BUCKET']}"

Trying to create a Glue session for the kernel.
Session Type: etl
Worker Type: G.1X
Number of Workers: 2
Session ID: c8680077-3f8c-489d-9933-cdb847fba005
Applying the following default arguments:
--glue_kernel_version 1.0.4
--enable-glue-datacatalog true
Waiting for session c8680077-3f8c-489d-9933-cdb847fba005 to get into ready status...
Session c8680077-3f8c-489d-9933-cdb847fba005 has been created.



---

<a id="section1"></a>
## 1. Batch Inference with the Serialized Model

Load the model trained in Part 1 and apply it in batch to the test dataset we saved there.

---
### Task 10: Batch Application

Load the model saved in Part 1. Apply it to the test data (also saved in Part 1) and evaluate using AUC.

In [2]:
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator


In [3]:
import os
model_loaded = PipelineModel.load(f"s3://{os.environ['WORK_BUCKET']}/word2vec_model/")
test_df_loaded = spark.read.parquet(f"s3://{os.environ['WORK_BUCKET']}/electronics_test/")
predictions = model_loaded.transform(test_df_loaded)

evaluator = BinaryClassificationEvaluator(labelCol="sentiment", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_loaded = evaluator.evaluate(predictions)

print(f"AUC del modelo cargado: {auc_loaded:.4f}")


AUC del modelo cargado: 0.9116


In [8]:
import pandas as pd
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


roc_df = predictions.select("probability", "sentiment").toPandas()


roc_df["prob_positive"] = roc_df["probability"].apply(lambda x: float(x[1]))


fpr, tpr, thresholds = roc_curve(roc_df["sentiment"], roc_df["prob_positive"])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Modelo de Sentiment (Test)")
plt.legend(loc="lower right")
plt.savefig("roc_curve.png") 


In [16]:
import os
import boto
s3 = boto3.client('s3')
s3.upload_file('roc_curve.png', os.environ['WORK_BUCKET'], 'images/roc_curve.png')



In [21]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

img = PILImage.open("roc_curve.png")
plt.imshow(img)
plt.axis('off')
plt.show()

In [22]:
import os
print(os.listdir())

['hsperfdata_spark', 'glue-default.conf', 'glue-override.conf', 'log4j2.properties', '.log4j2.properties.crc', 'exportCustomerEnvVariablesCmd', 'exportInternalEnvVariablesCmd', 'launchcmd', '5029492955850623067', 'blockmgr-e033ae39-ccd4-4817-8165-2c4969fab639', 'spark-d2baad95-16df-46c9-b667-8d72bed4010e', 'liblz4-java-911791017396719452.so.lck', 'liblz4-java-911791017396719452.so', 'hadoop-spark', 'roc_curve.png', 'glue-app-insights-logs', 'glue-exception-analysis-logs', 'glue_app_analyzer_rules', 'spark-event-logs', 'aws_glue_custom_connector_python', '.ICE-unix', '.Test-unix', '.X11-unix', '.XIM-unix', '.font-unix']
